# 02 Initial Model Screening

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`prepare_subset.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/prepare_subset.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`run_13_candidates.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/run_13_candidates.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/prepare_subset.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd6263d38f95793081d9e8d60473f685ec30c1357'}, '03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/run_13_candidates.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '0db3a679ccd042a93e752567efe7a976f180f496'}}

## prepare_subset

원본: [03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/prepare_subset.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/prepare_subset.py)  
Git blob: `d6263d38f95793081d9e8d60473f685ec30c1357`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Prepare common leakage-free subset for 13 baselines from reviewed ML04 v3.

Source 96k Training only. NO official Test and NO Hand_made at candidate screening.

CLI 인자 (정적 추출): `--official`

In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/prepare_subset.py
#!/usr/bin/env python3
"""Prepare common leakage-free subset for 13 baselines from reviewed ML04 v3.

Source 96k Training only. NO official Test and NO Hand_made at candidate screening.
"""
import hashlib,json,platform,argparse
from pathlib import Path
import numpy as np
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import StratifiedGroupKFold

ROOT=Path(__file__).resolve().parent
parser=argparse.ArgumentParser(description='Prepare group-separated 12k/3.2k preview dataset from v3.')
parser.add_argument('--official',type=Path,default=Path('02_data/datasets/MNIST_EMNIST_official_split_112000.npz'),
                    help='Path to the ORIGINAL verified official v3 NPZ')
args=parser.parse_args()
ROOT.mkdir(exist_ok=True)
INPUT=args.official
EXPECTED='0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879'
with INPUT.open('rb') as f: digest=hashlib.file_digest(f,'sha256').hexdigest()
assert digest==EXPECTED, f'Not the reviewed v3 file: {digest}'
with np.load(INPUT, allow_pickle=False) as data:
    x=data['X_train']
    y=data['y_train'].astype(np.int64)
    source=data['train_source_index'].astype(np.int64)
    split=data['train_source_split'].astype(np.int64)
    augmented=data['train_is_augmented']
assert x.shape==(96000,28,28) and len(y)==96000 and int(augmented.sum())==5349
groups=y * 1_000_000_000 + split*100_000_000 + source
skf=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
tidx,vidx=next(skf.split(x,y,groups))
assert len(tidx)==76800 and len(vidx)==19200
assert not set(groups[tidx]).intersection(groups[vidx])

def choose(indices,n_per_class,seed):
    rng=np.random.default_rng(seed)
    result=[]
    for cls in range(16):
        ids=indices[y[indices]==cls]
        assert len(ids)>=n_per_class,(cls,len(ids))
        result.extend(rng.choice(ids,size=n_per_class,replace=False).tolist())
    return np.asarray(sorted(result),dtype=np.int64)

train_idx=choose(tidx,750,42)
val_idx=choose(vidx,200,43)
assert len(train_idx)==12000 and len(val_idx)==3200
assert not set(groups[train_idx]).intersection(groups[val_idx])
assert all(np.bincount(y[train_idx],minlength=16)==750)
assert all(np.bincount(y[val_idx],minlength=16)==200)
# Small shape-preserving input to minimize memory overhead in evaluation subprocesses.
np.savez_compressed(ROOT/'common_subset.npz',X_train=x[train_idx],y_train=y[train_idx],X_val=x[val_idx],y_val=y[val_idx],train_idx=train_idx,val_idx=val_idx)
meta={'source_path':'02_data/datasets/MNIST_EMNIST_official_split_112000.npz',
      'source_sha256':digest,'source_training_count':96000,'source_official_test_count_NOT_USED':16000,
      'full_training_part':int(len(tidx)),'full_validation_part':int(len(vidx)),
      'screening_train_n':int(len(train_idx)),'screening_val_n':int(len(val_idx)),
      'per_class_train':750,'per_class_validation':200,'validation_group_leakage':0,
      'split':'StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42) first fold',
      'subsample_train_seed':42,'subsample_validation_seed':43,'classes':'0-9,A-F',
      'sample_shape':'(28,28)','sklearn_version':sklearn_version,'numpy_version':np.__version__,
      'python_version':platform.python_version(),
      'disclaimer':'Early 13-model screening only. No official Test or Hand-made data exposed.'}
(ROOT/'common_data_metadata.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(meta,indent=2,ensure_ascii=False),flush=True)

## run_13_candidates

원본: [03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/run_13_candidates.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/run_13_candidates.py)  
Git blob: `0db3a679ccd042a93e752567efe7a976f180f496`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Run 13 W01 classifier candidates with scikit-learn defaults on a common split.

Usage: python run_13_candidates.py --all --timeout 95
       python run_13_candidates.py --model 1

13 entries (SVC RBF representing default SVC; OvR logistic requires wrapper).
Random_state=42 is set only where available for reproducibility, all other
estimator hyperparameters remain their installed sklearn defaults.

CLI 인자 (정적 추출): `--all`, `--model`, `--timeout`

In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/experiment1_13_candidates/run_13_candidates.py
#!/usr/bin/env python3
"""Run 13 W01 classifier candidates with scikit-learn defaults on a common split.

Usage: python run_13_candidates.py --all --timeout 95
       python run_13_candidates.py --model 1

13 entries (SVC RBF representing default SVC; OvR logistic requires wrapper).
Random_state=42 is set only where available for reproducibility, all other
estimator hyperparameters remain their installed sklearn defaults.
"""
from __future__ import annotations
import argparse,csv,json,os,platform,subprocess,sys,time,warnings
from pathlib import Path
import numpy as np
from sklearn import __version__ as SKVERSION
from sklearn.linear_model import LogisticRegression,SGDClassifier,Perceptron
from sklearn.svm import LinearSVC,SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,AdaBoostClassifier,HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report

ROOT=Path(__file__).resolve().parent
CANDIDATES={
 1:('Logistic Regression (OvR)',True,lambda:OneVsRestClassifier(LogisticRegression(random_state=42))),
 2:('Softmax Regression',True,lambda:LogisticRegression(random_state=42)),
 3:('SGDClassifier',True,lambda:SGDClassifier(random_state=42)),
 4:('LinearSVC',True,lambda:LinearSVC(random_state=42)),
 5:('SVC (RBF, default)',True,lambda:SVC()),
 6:('KNeighborsClassifier (KNN)',True,lambda:KNeighborsClassifier()),
 7:('DecisionTreeClassifier',False,lambda:DecisionTreeClassifier(random_state=42)),
 8:('RandomForestClassifier',False,lambda:RandomForestClassifier(random_state=42)),
 9:('ExtraTreesClassifier',False,lambda:ExtraTreesClassifier(random_state=42)),
 10:('AdaBoostClassifier',False,lambda:AdaBoostClassifier(random_state=42)),
 11:('HistGradientBoostingClassifier',False,lambda:HistGradientBoostingClassifier(random_state=42)),
 12:('Perceptron',True,lambda:Perceptron(random_state=42)),
 13:('MLPClassifier',True,lambda:MLPClassifier(random_state=42)),
}

def estimator(i):
    name,scale,build=CANDIDATES[i]
    model=build()
    if scale:
        model=Pipeline([('scale_0_1',FunctionTransformer(lambda x:x/255.0,validate=False)),('model',model)])
    return model

def run_model(i):
    name,scale,_=CANDIDATES[i]
    with np.load(ROOT/'common_subset.npz',allow_pickle=False) as f:
        xtr=f['X_train'].reshape(-1,784).astype('float32')
        ytr=f['y_train'].astype(int)
        xv=f['X_val'].reshape(-1,784).astype('float32')
        yv=f['y_val'].astype(int)
    clf=estimator(i)
    t=time.perf_counter()
    with warnings.catch_warnings(record=True) as warns:
        warnings.simplefilter('always')
        clf.fit(xtr,ytr)
    fit_time=time.perf_counter()-t
    t=time.perf_counter()
    yp=clf.predict(xv)
    predict_time=time.perf_counter()-t
    e=clf.named_steps['model'] if isinstance(clf,Pipeline) else clf
    iterations=getattr(e,'n_iter_',None)
    if isinstance(iterations,np.ndarray): iterations=iterations.tolist()
    if isinstance(iterations,np.integer): iterations=int(iterations)
    diag_idx=np.arange(0,len(xtr),max(1,len(xtr)//1600))[:1600]
    train_probe_accuracy=float(accuracy_score(ytr[diag_idx],clf.predict(xtr[diag_idx])))
    result={'id':i,'model':name,'status':'completed','train_n':int(len(xtr)),
       'validation_n':int(len(xv)), 'accuracy':float(accuracy_score(yv,yp)),
       'macro_f1':float(f1_score(yv,yp,average='macro',labels=list(range(16)),zero_division=0)),
       'train_probe_n':len(diag_idx),'train_probe_accuracy':train_probe_accuracy,
       'fit_seconds':round(fit_time,6),'predict_seconds':round(predict_time,6),
       'predict_ms_per_image':round(predict_time*1000/len(xv),6),
       'scaling':'x/255' if scale else 'none (raw 0-255 pixels)',
       'n_iter':iterations,
       'warnings':[f'{w.category.__name__}: {str(w.message)}'[:800] for w in warns],
       'params':str(clf.get_params(deep=True)),
       'confusion_matrix':confusion_matrix(yv,yp,labels=list(range(16))).tolist(),
       'per_class':classification_report(yv,yp,labels=list(range(16)),target_names=list('0123456789ABCDEF'),output_dict=True,zero_division=0),
       'versions':{'python':platform.python_version(),'numpy':np.__version__,'scikit-learn':SKVERSION}}
    (ROOT/f'candidate_{i:02d}.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f'DONE {i:02d} {name} acc={result["accuracy"]:.4f} macroF1={result["macro_f1"]:.4f} fit={fit_time:.3f}s predict={predict_time:.3f}s',flush=True)

def aggregate():
    rows=[]
    for i,(name,scale,_) in CANDIDATES.items():
        path=ROOT/f'candidate_{i:02d}.json'
        rows.append(json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'id':i,'model':name,'status':'not_run'})
    columns=['id','model','status','train_n','validation_n','accuracy','macro_f1','fit_seconds',
            'predict_seconds','predict_ms_per_image','scaling','train_probe_accuracy','n_iter','warnings','reason']
    with (ROOT/'results_13_models.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=columns);w.writeheader()
        for d in rows:
            row={key:d.get(key,'') for key in columns}
            row['warnings']=' | '.join(d.get('warnings',[])) if isinstance(d.get('warnings',[]),list) else str(d.get('warnings',''))
            w.writerow(row)
    m=json.loads((ROOT/'common_data_metadata.json').read_text(encoding='utf-8'))
    (ROOT/'results_13_models.json').write_text(json.dumps({'metadata':m,'models':rows},ensure_ascii=False,indent=2),encoding='utf-8')

def main():
    pa=argparse.ArgumentParser();pa.add_argument('--all',action='store_true');pa.add_argument('--model',type=int,choices=list(CANDIDATES));pa.add_argument('--timeout',type=int,default=95)
    a=pa.parse_args()
    if a.model:run_model(a.model)
    elif a.all:
        for i,(name,scale,_) in CANDIDATES.items():
            if (ROOT/f'candidate_{i:02d}.json').exists():continue
            print(f'START {i:02d} {name}',flush=True)
            t=time.perf_counter()
            env={**os.environ,'OPENBLAS_NUM_THREADS':'2','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','NUMEXPR_NUM_THREADS':'2'}
            p=subprocess.run(['timeout','--signal=TERM','--kill-after=4s',f'{a.timeout}s',sys.executable,str(Path(__file__).resolve()),'--model',str(i)],
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env,cwd=ROOT)
            elapsed=time.perf_counter()-t
            print(p.stdout[-2000:],flush=True)
            if p.returncode !=0:
                status='timeout' if p.returncode==124 else 'failed'
                reason=f'Exit {p.returncode} elapsed {elapsed:.1f}s with max {a.timeout}s (see execution_log.txt)'
                (ROOT/f'candidate_{i:02d}.json').write_text(json.dumps({'id':i,'model':name,'status':status,'reason':reason,
                  'elapsed_seconds':elapsed,'trace_tail':p.stdout[-2000:]},ensure_ascii=False,indent=2),encoding='utf-8')
                print(f'{status.upper()} {i:02d} {name} {reason}',flush=True)
            aggregate()
            print(f'PROGRESS {i}/13',flush=True)
        aggregate();print('ALL 13 candidates attempted',flush=True)
    else:pa.error('Specify --all or --model')

if __name__=='__main__':main()

## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")